# Clean `emr_cycle_outcome`

A first-pass cleaning outcome information.

What this notebook does, in order:
1. **Profile** every column (how full it is, how many distinct values, its type)
2. **Drop** columns that are completely empty
3. **Review** columns that hold only a single value
4. **Merge** the oocyte dataframes (by absolute_oocyteid)
5. **Tidy** column types
6. **Save** a cleaned copy + keep an audit note of what changed

The same `profile()` function works on the other `emr_*` tables too, so you can reuse this pattern.

> **Before running:** finish the venv setup in this repo and install the packages — open a terminal and run `pip install pandas pyarrow`. The `requirements.txt` alongside this notebook lists everything.
>
> **Important:** keep your raw patient data in a `data/` folder that is git-ignored. Don't commit EMR exports to GitHub, even a private repo.

## Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import import_ipynb

import helper_functions

In [2]:
# --- emr_oocyte ---
cycle_outcome_PATH = Path("../data/emr_cycle_outcome.csv")

# loading data
outcome_df = pd.read_csv(cycle_outcome_PATH)

In [3]:
# list all columns of the oocyte_df
display(outcome_df.columns)

Index(['id', 'addedby', 'addedon', 'deletedby', 'deletedon', 'cycleid',
       'patid', 'cycle_outcome', 'pregnancy_date', 'ultrasound_performed',
       'ultrasound_performedby', 'reduction_performed',
       'reduction_performedby', 'reduction_number', 'source',
       'max_fetal_hearts', 'monochorionic_twin', 'num_fetus', 'complication',
       'note', 'preg_outcome', 'preg_outcome_date', 'preg_method', 'preg_note',
       'preg_other', 'reduction', 'ultrasound', 'ultrasound_performedby_other',
       'reduction_performedby_other', 'edd', 'pregnancy_date_2',
       'pregnancy_result', 'pregnancy_result_2', 'ultrasound_2',
       'ultrasound_performedby_2', 'ultrasound_performed_2',
       'ultrasound_performedby_other_2', 'max_fetal_hearts_2', 'covid_cancel',
       'importid', 'hospitalization_occurred', 'cycle_occurred_intended',
       'contact_attempt', 'advers_events_from_device', 'outcome_date'],
      dtype='str')

In [4]:
#setting display options to show all columns and full width
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)        # Allow full width

## 1. Profile every column

One row per column so you can see at a glance what's worth keeping.

In [5]:
# Already know there are many all NULL columns, so dropping those first
print(outcome_df.shape)
outcome_df = outcome_df.dropna(axis=1, how="all")
print(outcome_df.shape)

(8984, 45)
(8984, 32)


In [6]:
# now profiling the dataframes to get a better understanding of the data and dropping unninformative columns

outcome_prof = helper_functions.profile(outcome_df)
outcome_df = helper_functions.drop_empty_and_constant_columns(outcome_df, outcome_prof)
print(outcome_prof)

Dropping 1 columns
                                non_null  nulls  distinct    dtype  pct_null  \
id                                  8984      0      8984      str       0.0   
addedby                             8984      0        56    int64       0.0   
addedon                             8984      0      8963      str       0.0   
cycleid                             8984      0      8983      str       0.0   
patid                               8984      0      3454    int64       0.0   
cycle_outcome                       8461    523         7      str       5.8   
hospitalization_occurred            8398    586         2  float64       6.5   
covid_cancel                        8090    894         2  float64      10.0   
cycle_occurred_intended             6300   2684         2  float64      29.9   
num_fetus                           1864   7120         4  float64      79.3   
ultrasound                          1677   7307         2      str      81.3   
ultrasound_performed 

## 2. Cleaning data


In [7]:
display(outcome_df.sample(20))

,id,addedby,addedon,cycleid,patid,cycle_outcome,ultrasound_performed,ultrasound_performedby,reduction_performedby,source,max_fetal_hearts,monochorionic_twin,num_fetus,complication,note,preg_outcome,preg_outcome_date,preg_method,preg_note,preg_other,ultrasound,ultrasound_performedby_other,ultrasound_2,ultrasound_performedby_2,ultrasound_performed_2,ultrasound_performedby_other_2,max_fetal_hearts_2,covid_cancel,hospitalization_occurred,cycle_occurred_intended,outcome_date
8123,aa40f2a9-7ccf-4a66-b59c-33554c2f94bd,35,2025-04-08 11:46:29.458303-07,f15a3712-0cba-4e37-953c-21993b314e0f,14021,Clinical Intrauterine Gestation,2025-04-08 11:07:17,90.0,NaN,Patient (verbal only),1.0,No,1.0,NaN,NaN,Live birth,2025-12-05 00:00:00,Vaginal,NaN,NaN,Yes,NaN,Yes,90.0,2025-04-22 10:46:09,NaN,1.0,NaN,0.0,0.0,2025-12-15 00:00:00
45,652ad334-fbc4-4801-a2b6-adb9c2da89cc,54,2021-12-07 12:31:40.506-08,6cb672bf-287a-47a8-9a38-ac67ae22386d,24210,Not Pregnant,NaN,54.0,54.0,NaN,NaN,NaN,NaN,NaN,CD1 today 12/7/21 /lc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,54.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN
5176,e72c93d1-94dd-40f4-b109-eb450b8b4302,14,2024-07-16 14:56:08.115032-07,75c38438-a487-40a2-a2c3-752549ab2fdc,51455,Not Pregnant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN
3885,953d90ab-2d9a-4c88-a714-50171401bd44,53,2023-11-10 10:19:20.028804-08,cc9910a4-701f-4458-915e-cc9dcc8f2f48,41323,Not Pregnant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1.0,NaN
8462,6b1ceec2-5704-459a-95c0-6788261c2017,2025,2026-04-20 10:53:44.042101-07,02b5f118-c5fe-4f68-815d-44d7ea8bb17c,130854,Not Pregnant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1.0,NaN
1121,1cec9242-0dbd-4971-b92a-35c3647bac81,60,2022-08-05 10:24:30.5371-07,72463311-da77-4672-9c46-48238ee28869,16350,Not Pregnant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN
3241,d60dd2f0-8839-4bd2-8652-6ffcbbb7edd0,144,2023-09-27 12:51:47.218367-07,2e5ff742-4267-4bb0-820f-86e3dc586e99,30239,Not Pregnant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN
768,ab990b3e-ddca-4116-abf9-35aa0f6c5286,144,2023-01-18 12:04:20.507094-08,dc15f2fc-abba-4321-afdb-fe4bd873168c,37103,Not Pregnant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN
7753,3d1d1157-5bdc-40f0-b5f4-b39a8401c764,2025,2025-10-30 14:06:15.869587-07,72e109c5-d04d-4ced-a31c-a1ae8c53df81,55183,Froze All,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1.0,NaN
4291,0b4f9224-4ea5-4c15-a526-42ee6d6bcb05,44,2024-02-04 12:08:21.938871-08,74bf5bb5-e1f3-4eb7-922e-62f65d6d8f9b,41455,Not Pregnant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1.0,NaN


## 3. Keeping subset of columns


In [10]:
# selecting columns to keep for oocyte_df
outcome_keep_cols = ["patid", "cycleid", "cycle_outcome", "max_fetal_hearts", 
                    "max_fetal_hearts_2", "num_fetus", "complication", "note", 
                    "preg_outcome"]

In [11]:
# filter the dataframes to keep only those columns
# -------- outcome_df --------
outcome_df = outcome_df[outcome_keep_cols]
print(f"\nAfter filtering to {len(outcome_keep_cols)} columns: {outcome_df.shape[1]} columns")


After filtering to 9 columns: 9 columns


## 4. Saving as csv

In [12]:
# save as csv
outcome_df.to_csv("../data/emr_cycle_outcome_processed.csv", index=False)